In [1]:
import pandas as pd
import stlearn as st
import anndata as ad
import numpy as np
from PIL import Image
import seaborn as sns
import scanpy as sc
import squidpy as sq
import sopa
import sopa.spatial
import monkeybread as mb
from spatialdata._logging import logger
from matplotlib import pyplot as plt
heatmap_kwargs = {"vmax": 40, "cmap": sns.cm.rocket_r, "cbar_kws": {'label': 'Mean hop distance'}}

/home/uqxtan9/micromamba/envs/spatialdata/lib/python3.10/site-packages/stlearn/tools/microenv/cci/het.py:192: NumbaDeprecationWarning: The keyword argument 'nopython=False' was supplied. From Numba 0.59.0 the default is being changed to True and use of 'nopython=False' will raise a warning as the argument will have no effect. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @jit(parallel=True, nopython=False)


In [10]:
import subprocess
from pathlib import Path
system = subprocess.check_output(["hostname", "-s"]).decode("utf-8").strip()
BASE_PATH_ = Path()
if "bun" in system:
    BASE_PATH_ = Path("/scratch/project_mnt/S0010/Xiao/Q1851/Xiao/")
elif "imb-quan-gpu" in system:
    BASE_PATH_ = Path("/home/uqxtan9/Q1851/Xiao/")


BASE_PATH = BASE_PATH_ / "Working_project/MB"
DATA_PATH = BASE_PATH / "Xenium_Brain"
XENIUM_RAW_PATH = DATA_PATH / "Xenium_RAW"
MALDI_RAW_PATH = DATA_PATH / "MALDI_RAW/imzml_file"
MALDI_PROCESSED = BASE_PATH / "MALDI_PROCESSED"
PROCESSED = BASE_PATH / "PROCESSED"
PROCESSED.mkdir(exist_ok=True, parents=True)
OUT_PATH = BASE_PATH / "PLOTS" / "Xenium"
OUT_PATH.mkdir(exist_ok=True, parents=True)
QC_PATH = OUT_PATH / "QC"
QC_PATH.mkdir(exist_ok=True, parents=True)
CLS_PATH = OUT_PATH / "CLUSTERING"
CLS_PATH.mkdir(exist_ok=True, parents=True)
CCI_PATH = OUT_PATH / "CCI"
CCI_PATH.mkdir(exist_ok=True, parents=True)
DE_PATH = MALDI_PROCESSED / "DE"
DE_PATH.mkdir(exist_ok=True, parents=True)

In [3]:
file_id_ls = ["C1", "C2", "T1", "T2"]
library_id_ls = ["Ctrl_1", "Ctrl_2", "Treated_1", "Treated_2"]

In [4]:
MALDI_adata = {}
for library_id in library_id_ls:
    adata = ad.read_h5ad(MALDI_PROCESSED / f"{library_id}_MALDI_adata_dist.h5ad")
    MALDI_adata[library_id] = adata

In [13]:
for library_id, adata in MALDI_adata.items():
    logger.info(library_id)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    sc.tl.rank_genes_groups(adata, groupby="cell_type", method="wilcoxon")
    df_de = sc.get.rank_genes_groups_df(adata, group=None, pval_cutoff=0.05)
    df_de.to_csv(MALDI_PROCESSED / f"{library_id}_DE_celltype.csv")

INFO     Ctrl_1                                                                                                    
INFO     Ctrl_2                                                                                                    
INFO     Treated_1                                                                                                 
INFO     Treated_2                                                                                                 


In [16]:
for library_id, adata in MALDI_adata.items():
    df_de = pd.read_csv(MALDI_PROCESSED / f"{library_id}_DE_celltype.csv", index_col=0)
    for cell_type in df_de["group"].unique():
        df_out = df_de[df_de["group"] == cell_type]
        df_out = df_out[["names", "pvals_adj", "logfoldchanges"]]
        df_out.columns = ["m.z", "p.value", "t.score"]
        df_out["m.z"] = df_out["m.z"].str.replace("mz-", "")
        df_out.to_csv(DE_PATH / f"{library_id}_DE_{cell_type}.txt",index=False,sep="\t")

In [17]:
for library_id, adata in MALDI_adata.items():
    df_de = pd.read_csv(MALDI_PROCESSED / f"{library_id}_DE_celltype.csv", index_col=0)
    df_out = df_de
    df_out = df_out[["names", "pvals_adj", "logfoldchanges"]]
    df_out.columns = ["m.z", "p.value", "t.score"]
    df_out["m.z"] = df_out["m.z"].str.replace("mz-", "")
    df_out.to_csv(DE_PATH / f"{library_id}_DE_all.txt",index=False,sep="\t")

/tmp/ipykernel_3434044/816171642.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_out["m.z"] = df_out["m.z"].str.replace("mz-", "")
/tmp/ipykernel_3434044/816171642.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_out["m.z"] = df_out["m.z"].str.replace("mz-", "")
/tmp/ipykernel_3434044/816171642.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pa

In [15]:
df_out["m.z"] = df_out["m.z"].str.replace("mz-", "")
df_out["m.z"]

4446    783.564840016871
4447    757.545661862516
4448    804.542518000466
4449    725.552776914166
4450    755.533268072252
              ...       
4853     422.92455884066
4854    876.576356024091
4855    771.513226285263
4856    848.545591622295
4857    268.101010738467
Name: m.z, Length: 412, dtype: object

In [ ]:
DE_PATH